# Exercises (Student) - MCP Client with LLM

In [1]:
!pip install -q mcp nest_asyncio requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.8 MB/s eta 0:00:00


In [2]:

import os
from pathlib import Path
MCP_HTTP_TOKEN = os.getenv("MCP_HTTP_TOKEN", "devtoken123")
USE_REAL_LLM = False  # flip True if GITHUB_TOKEN is set


In [3]:
import asyncio
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()


In [22]:
%%writefile server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("DemoServer")

#To-Do: Add decorator to register add as a tool
@mcp.tool()
def add(a: int, b: int) -> int:
    "Add two numbers."
    return a + b

# TODO (optional exercise): add multiply(a, b) here if you want an extra tool
@mcp.tool()
def multiply (a:int, b:int) -> int :
  "Multiply two numbers"
  return a * b

#To-Do: Add decorator to register greet as a tool
@mcp.tool()
def greet(name: str) -> str:
    "Return a greeting string."
    return f'Hello {name}'

if __name__ == "__main__":
    mcp.run()


Overwriting server.py


## Exercise 1 (provide answer)

#To-Do: Why is STDIO transport simple for local MCP dev compared to HTTP?

Pour le développement local, c’est la solution la plus simple : l’interface de ligne de commande MCP lance votre serveur en tant que sous-processus ; les messages circulent via stdin/stdout — aucune configuration de ports n’est requise.

## Exercise 2

In [18]:
!python -m mcp run server.py -q

/usr/bin/python3: No module named mcp.__main__; 'mcp' is a package and cannot be directly executed


In [23]:

import asyncio
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

async def ex2_connect():
    #To-Do: Create StdioServerParameters with command to run server.py
    params = StdioServerParameters(
        command="mcp",
        args=["run", str('server.py')],
        env=None,
    )
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()


In [24]:
# In a new cell
await ex2_connect()
print("Exercise 2: OK (connected and initialized)")


Exercise 2: OK (connected and initialized)


## Exercise 3

In [26]:
async def ex3_list():
    #To-Do: Create StdioServerParameters with command to run server.py
    params = StdioServerParameters(
        command="mcp",
        args=["run", str('server.py')],
        env=None,
    )
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            resources = await session.list_resources()
            print("RESOURCES:", resources)
            tools = await session.list_tools()
            for t in tools.tools:
                print(t.name, t.inputSchema.get("properties", {}))


In [27]:
await ex3_list()

RESOURCES: meta=None nextCursor=None resources=[]
add {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
multiply {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
greet {'name': {'title': 'Name', 'type': 'string'}}


## Exercise 4

#To-Do: Explain how the conversion to llm tool happens in MCP server code ?

La plupart des modèles de chat acceptent un tableau « functions/tools » avec des schémas JSON. Donc nous allons adapter les métadonnées MCP à cette structure. De plus,  elle fournit au LLM un contrat typé précis afin qu’il puisse renvoyer un contrat valide tool_callsque votre client peut exécuter en toute sécurité.

In [28]:

def convert_to_llm_tool(tool):
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description or "mcp tool",
            "parameters": {
                "type": "object",
                "properties": tool.inputSchema.get("properties", {}),
                "required": tool.inputSchema.get("required", []),
            },
        },
    }


## Exercise 5

**Plan & execute:** Use stub (or real) LLM to propose `tool_calls`, then execute them and print results for a prompt like “Add 2 to 20.”

In [29]:

import asyncio
import json
import nest_asyncio
from typing import Any, Dict, List
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

def call_llm(prompt: str, functions: List[Dict[str, Any]], use_real: bool = False):
    if not use_real:
        return stub_plan(prompt, functions)
    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use stub planner.")
    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential
    client = ChatCompletionsClient("https://models.inference.ai.azure.com", AzureKeyCredential(token))
    resp = client.complete(
        model="gpt-4o",
        messages=[{"role": "system", "content": "Plan MCP tool calls."},{"role": "user", "content": prompt}],
        tools=functions,
        temperature=0,
        max_tokens=400,
    )
    calls = []
    msg = resp.choices[0].message
    for tc in msg.tool_calls or []:
        args = tc.function.arguments
        args_json = json.loads(args) if isinstance(args, str) else args
        calls.append({"name": tc.function.name, "args": args_json})
    return calls


In [53]:
import re

def stub_plan(prompt: str, functions):
    numbers = list(map(int, re.findall(r'\d+', prompt)))

    if "add" in prompt.lower() and len(numbers) >= 2:
        return [{
            "name": "add",
            "args": {
                "a": numbers[0],
                "b": numbers[1]
            }
        }]

    elif "multiply" in prompt.lower() and len(numbers) >= 2:
        return [{
            "name": "multiply",
            "args": {
                "a": numbers[0],
                "b": numbers[1]
            }
        }]

    elif "greet" in prompt.lower():
        return [{
            "name": "greet",
            "args": {
                "name": "User"
            }
        }]

    return []

In [39]:
import sys

In [54]:
async def ex5_run(prompt: str = "Add 2 to 20"):
    params = StdioServerParameters(
            command="mcp",
            args=["run", str('server.py')],
            env=None,
            )
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            tools = await session.list_tools()
            functions = [convert_to_llm_tool(t) for t in tools.tools]
            calls = call_llm(prompt, functions, use_real=USE_REAL_LLM)
            print("tool_calls:", calls)
            for call in calls:
                result = await session.call_tool(call["name"], arguments=call["args"])
                print("result:", [getattr(c, "text", str(c)) for c in result.content])


In [56]:
await ex5_run("Add 21 to 23")

tool_calls: [{'name': 'add', 'args': {'a': 21, 'b': 23}}]
result: ['44']


## Optional - add multiply(a, b) and rerun

In [57]:
await ex5_run("Multiply 22 to 28")

tool_calls: [{'name': 'multiply', 'args': {'a': 22, 'b': 28}}]
result: ['616']


In [58]:
await ex5_run("greet cyriac")

tool_calls: [{'name': 'greet', 'args': {'name': 'User'}}]
result: ['Hello User']
